## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-02-19-51-54 +0000,nyt,He Pioneered a Groundbreaking Idea and Was Met...,https://www.nytimes.com/2026/11/02/climate/wil...
1,2026-11-02-19-50-01 +0000,nypost,JD Vance rallies GOP House candidate Anthony C...,https://nypost.com/2026/11/02/us-news/vice-pre...
2,2026-11-02-19-49-51 +0000,nyt,Are Freezers Being Hacked? Europe and the U.S....,https://www.nytimes.com/2026/11/02/business/cy...
3,2026-11-02-19-49-45 +0000,nyt,Here’s the latest.,https://www.nytimes.com/live/2026/11/02/us/mid...
4,2026-11-02-19-44-31 +0000,nyt,20-Year-Old Nathalia Filgueiras Was a Pilot on...,https://www.nytimes.com/2026/11/02/us/nathalia...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2911/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
79,trump,30
84,election,16
133,race,14
132,senate,13
33,florida,13


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
257,2026-11-02-05-08-00 +0000,wsj,Rep. Haley Stevens stunned Democrats on Sunday...,https://www.wsj.com/politics/elections/how-rep...,122
181,2026-11-02-11-55-00 +0000,wsj,As the midterms bumble and stumble and fumble ...,https://www.wsj.com/politics/trumps-monday-spe...,88
84,2026-11-02-17-38-10 +0000,nypost,Trump takes fight over 20K California mail-in ...,https://nypost.com/2026/11/02/us-news/trump-ta...,87
56,2026-11-02-18-27-33 +0000,cbc,"Trump administration ramps up midterm battles,...",https://www.cbc.ca/news/world/trump-administra...,87
13,2026-11-02-19-34-04 +0000,chitri,President Donald Trump spends day before midte...,https://www.chicagotribune.com/2026/11/02/dona...,87


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
257,122,2026-11-02-05-08-00 +0000,wsj,Rep. Haley Stevens stunned Democrats on Sunday...,https://www.wsj.com/politics/elections/how-rep...
181,88,2026-11-02-11-55-00 +0000,wsj,As the midterms bumble and stumble and fumble ...,https://www.wsj.com/politics/trumps-monday-spe...
316,62,2026-11-01-20-00-00 +0000,wsj,"In the Florida governor’s race, Democrat David...",https://www.wsj.com/politics/elections/democra...
84,57,2026-11-02-17-38-10 +0000,nypost,Trump takes fight over 20K California mail-in ...,https://nypost.com/2026/11/02/us-news/trump-ta...
104,49,2026-11-02-16-46-44 +0000,nypost,US and Iran resume ‘very intensive’ peace talk...,https://nypost.com/2026/11/02/world-news/us-an...
180,42,2026-11-02-11-59-01 +0000,nypost,Hackers breach emergency alert apps in Ukraine...,https://nypost.com/2026/11/02/world-news/hacke...
103,36,2026-11-02-16-48-01 +0000,nypost,Doomed Learjet co-pilot Nathalia Filgueiras te...,https://nypost.com/2026/11/02/us-news/doomed-l...
42,34,2026-11-02-18-49-55 +0000,nypost,"Karen Bass, Xavier Becerra make final LA campa...",https://nypost.com/2026/11/02/us-news/karen-ba...
80,33,2026-11-02-17-50-45 +0000,chitri,Aldermen put $75 million in parking meter mone...,https://www.chicagotribune.com/2026/11/02/alde...
114,32,2026-11-02-16-04-42 +0000,nypost,DC federal judge tosses case against FBI offic...,https://nypost.com/2026/11/02/us-news/dc-feder...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
